# Timepoint Separation Models — Email‑Guided Baseline

**Objective:** Build a simple, interpretable model that maximizes separation between timepoints (v1 vs v2 for Melbourne; ses‑1 vs ses‑3 for Campinas), following the supervisor’s email guidance.

**Key idea:** Start with **neuroimaging‑only composite scores**, then optionally add demographic/disease‑history variables and **limited interaction terms**.


## Section 0 — Setup

In [1]:
import os, math, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score

warnings.filterwarnings("ignore")

# Ensure we run from repo root regardless of where Jupyter was launched
REPO_ROOT = Path("..").resolve()
os.chdir(REPO_ROOT)
print("Working directory:", Path.cwd())

DEMO_MODE = False
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


Working directory: /Users/robertwang/Documents/New project/baseline_composite_biomakers


## Section 1 — Load Updated Merged Data

In [2]:
data_path = Path("data/processed/combined_frda_merged.csv")
assert data_path.exists(), "Run the merge notebook first to generate combined_frda_merged.csv"

df = pd.read_csv(data_path)
print("Loaded merged dataset:", df.shape)
print(df[["subject"]].head())


Loaded merged dataset: (54, 85)
         subject
0  sub-melfrd003
1  sub-melfrd004
2  sub-melfrd005
3  sub-melfrd006
4  sub-melfrd008


## Section 2 — Define Features and Phases

In [3]:
# Neuroimaging features (structural + diffusion + spinal cord if present)
structural = [
    "scp_vol", "mcp_vol", "icp_vol",
    "midbrain_vol", "pons_vol", "medulla_vol",
    "cb_ant_vol", "cb_sup_post_vol", "cb_inf_post_vol", "cb_floc_vol", "cb_vermis_vol",
]

diffusion = [
    "scp_fa", "mcp_fa", "icp_fa",
    "scp_md", "mcp_md", "icp_md",
    "scp_ad", "mcp_ad", "icp_ad",
    "scp_rd", "mcp_rd", "icp_rd",
]

spinal = [c for c in df.columns if "csa" in c.lower() or "ecc" in c.lower()]

# Demographic / disease-history
meta = [
    "age_at_scan", "sex",
    "disease_duration", "age_at_onset",
    "GAA1", "GAA2",
]

# Filter to columns that exist in the merged dataset
structural = [c for c in structural if c in df.columns]
diffusion = [c for c in diffusion if c in df.columns]
meta = [c for c in meta if c in df.columns]

neuro = structural + diffusion + spinal

print("Neuroimaging features:", len(neuro))
print("Demographic features:", len(meta))


Neuroimaging features: 8
Demographic features: 4


## Section 3 — Build Timepoint Labels

In [4]:
# Melbourne: v1 vs v2
# Campinas: ses-1 vs ses-3

def timepoint_label(row):
    sess = str(row.get("session", ""))
    if sess in ["ses-1", "ses-2", "ses-3"]:
        if sess == "ses-1":
            return 0
        if sess == "ses-3":
            return 1
    # fallback on visit column if present
    if "visit" in row and not pd.isna(row["visit"]):
        return 0 if int(row["visit"]) == 1 else 1
    return np.nan

labels = df.apply(timepoint_label, axis=1)
df = df.assign(timepoint=labels)

df = df.dropna(subset=["timepoint"]).copy()
print("Rows with timepoint labels:", df.shape)


Rows with timepoint labels: (0, 86)


## Section 4 — Feature Phases

In [5]:
# Phase A: Neuroimaging only
# Phase B: Demographics only
# Phase C: Combined + limited interactions

phases = {
    "neuro_only": neuro,
    "demographics_only": meta,
    "combined": neuro + meta,
}

# Add limited interaction terms (neuro × disease_duration, neuro × age_at_onset)
for base in ["disease_duration", "age_at_onset"]:
    if base in df.columns:
        for f in neuro:
            name = f"{f}__x__{base}"
            df[name] = df[f] * df[base]
        phases["combined"] += [f"{f}__x__{base}" for f in neuro]

print("Phases defined:", {k: len(v) for k, v in phases.items()})


Phases defined: {'neuro_only': 8, 'demographics_only': 4, 'combined': 20}


## Section 5 — Nested CV with Linear Models

In [6]:
def run_nested_cv(X, y, groups):
    outer = GroupKFold(n_splits=5)

    # LDA with shrinkage + Logistic Regression (L2)
    models = {
        "LDA_shrink": (LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"), {}),
        "LogReg_L2": (LogisticRegression(max_iter=2000, solver="liblinear"), {"C": [0.01, 0.1, 1, 10]}),
    }

    results = []

    for name, (estimator, grid) in models.items():
        all_preds = []
        all_true = []

        for train_idx, test_idx in outer.split(X, y, groups):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            # Pipeline: scale then model
            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("model", estimator),
            ])

            if grid:
                gs = GridSearchCV(pipe, {f"model__{k}": v for k, v in grid.items()}, cv=3)
                gs.fit(X_train, y_train)
                pred = gs.predict(X_test)
            else:
                pipe.fit(X_train, y_train)
                pred = pipe.predict(X_test)

            all_preds.extend(pred)
            all_true.extend(y_test)

        acc = accuracy_score(all_true, all_preds)
        bacc = balanced_accuracy_score(all_true, all_preds)

        results.append({
            "model": name,
            "accuracy": acc,
            "balanced_accuracy": bacc,
        })

    return pd.DataFrame(results)


## Section 6 — Run Models for Each Phase

In [7]:
results = []

for phase, feats in phases.items():
    feats = [f for f in feats if f in df.columns]
    sub = df.dropna(subset=feats + ["timepoint"]).copy()

    X = sub[feats].values
    y = sub["timepoint"].values
    groups = sub["subject"].values

    print("Phase:", phase, "| N:", len(sub), "| Features:", len(feats))
    res = run_nested_cv(X, y, groups)
    res["phase"] = phase
    results.append(res)

results_df = pd.concat(results, ignore_index=True)
results_df


Phase: neuro_only | N: 0 | Features: 8


ValueError: Cannot have number of splits n_splits=5 greater than the number of samples: n_samples=0.

## Section 7 — Save Outputs

In [ ]:
out_dir = Path("results")
out_dir.mkdir(exist_ok=True)

results_df.to_csv(out_dir / "timepoint_metrics.csv", index=False)
print("Saved timepoint_metrics.csv")
